# 02 · Limpieza de datos

Preparo los datos para el modelo de riesgo de lesión. Parto de los CSV originales de `data/raw/`, aplico las decisiones tomadas en la exploración (`01-exploracion-datasetNBA.ipynb`) y guardo una tabla limpia en `data/nba_load.parquet`, que es la que carga `project-model.ipynb`.

Cada paso queda registrado con el número de filas que afecta.

In [3]:
#importación de herramientas y librerías necesarias para el análisis de datos
from pathlib import Path
import numpy as np #librería de cálculo númerico sobre la que funciona pandas
import pandas as pd

RAW = Path("..")/"data"/"raw"
OUT = Path("..")/"data"/"nba_load.parquet" #es donde se guardará el fichero limpio en project/data 
pd.set_option("display.max_columns", None)

### 0. CARGA DE DATOS

In [4]:
#Cargar el fichero de datos en bruto
player_stats = pd.read_csv(RAW/"playerStatistics.csv", low_memory=False) #low_memory=False para evitar el warning de memoria
players = pd.read_csv(RAW/"players.csv", low_memory=False)

#crear lista vacía y en cada paso se va a apuntar una línea con tres datos:
#- que se hizo, - cuantas filas afectó, - por qué se hizo. 
informe =[] #Esta es la lista vacía

print("Filas iniciales de player_stats:", len(player_stats))

Filas iniciales de player_stats: 1669922


### 1. LIMPIEZA DE DATOS (los de interés)

In [6]:
#Empezamos a limpiar el fechero y a filtrar los datos que me interesan

#PRIMERO 
#Fechas: solo quiero desde la temporada 2015-16 
player_stats["gameDate"] = pd.to_datetime(player_stats["gameDate"], errors="coerce") #convierto la columna de fechas a tipo fecha, si no puede convertirlo lo pone como NaT
n = len(player_stats) #guardo el número de filas inicial
player_stats = player_stats[player_stats["gameDate"] >= "2015-10-01"] #filtro las filas que cumplen la condición

#Añado a la lista de informe lo que se ha hecho:
informe.append(["Filtrado de fechas: partidos anteriores al 01/10/2015", n-len(player_stats), "Se han eliminado los partidos anteriores a la temporada 2015-16 (fuera de mi ventana de trabajo)"])
#-----------------------------------------------------------

#SEGUNDO
#Tipo de partido: solo quiero partidos de temporada regular y  playoffs. 
#Voy a unificar los nombres de los tipos de partido y quedarme con los que me interesen. 
mapa_tipo_partido = {
    "Regular Season": "Regular Season", "Playoffs": "Playoffs", "Play-in Tournament": "Play-in",
    "NBA Emirates Cup": "NBA Cup", "Emirates NBA Cup": "NBA Cup", "NBA Cup": "NBA Cup",
    "in-season-knockout": "NBA Cup", "Preseason": "Preseason", "Pre Season": "Preseason",
    "All-Star Game": "All-Star",
}
player_stats["gameType"] = player_stats["gameType"].map(mapa_tipo_partido)
n = len(player_stats)
player_stats = player_stats[player_stats["gameType"].isin(["Regular Season", "Playoffs", "Play-in", "NBA Cup"])]

#Añado a la lista de informe lo que se ha hecho:
informe.append(("No he inlcuido Pretemporada, All-Star y otros", n - len(player_stats), "No generan carga competitiva real"))
#-----------------------------------------------------------

#TERCERO
#Jugadores sin indetificar
n=len(player_stats)
player_stats = player_stats.dropna(subset=["personId"]) #elimino las filas que no tienen personId

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Eliminación de jugadores sin personId", n - len(player_stats), "No se puede identificar al jugador, no se puede analizar"))

player_stats["personId"] = player_stats["personId"].astype(int) #convierto a int la columna personId para poder hacer el merge con players

#-----------------------------------------------------------

print("Filas finales de player_stats:", len(player_stats))
player_stats["gameType"].value_counts() #veo cuántas filas hay de cada tipo de partido

Filas finales de player_stats: 366218


gameType
Regular Season    338439
Playoffs           24959
NBA Cup             1822
Play-in              998
Name: count, dtype: int64

In [7]:
pd.DataFrame(informe, columns=["Paso", "Filas afectadas", "Decisión y motivo"])

,Paso,Filas afectadas,Decisión y motivo
0,Filtrado de fechas: partidos anteriores al 01/...,1273392,Se han eliminado los partidos anteriores a la ...
1,"No he inlcuido Pretemporada, All-Star y otros",30312,No generan carga competitiva real
2,Eliminación de jugadores sin personId,0,"No se puede identificar al jugador, no se pued..."


### 2. CONVERTIR MINUTOS

Ahora voy a convertir ´numMinutes´ de texto a número, incluidos los valores en formato mm:ss.

La **razón** es que es mi variable de carga, mientras sea texto no la puedo sumar y la carga de 14 días es justamente una suma de minutos.

**Lo que espero ver:**

- **Filas covertidas**: algo menos de 2.097, porque algunas eran de pretemporada o enteriores al 2015 y ya las he quitado anteriormente. 
- ´describe()´: un resumen de la columna. El maximo deberia rondar los 60 minutos ya que un partido dura eso con prórrogas aproximadamente y el mínimo, 0. Si hay valores negativos hay algo mal.  

In [9]:
#PRIMERO: convertir la columna de minutos jugados a decimal.
#Trato la columna como texto y localizo las filas con formato mm:ss
minutos_txt = player_stats["numMinutes"].astype("string")
es_mmss = minutos_txt.str.contains(":", na=False) #MARCA TRUE si la fila tiene formato mm:ss, FALSE si no lo tiene o es NaN

#SEGUNDO: convierto las filas normales a número
#Las filas normales ("22.0") se convierten directamente a número
minutos = pd.to_numeric(minutos_txt.where(~es_mmss), errors="coerce")
#~es_mmss significa "no es mm:ss", el simbolo ~ convierte el True en False y viceversa. La función where() devuelve los valores de la serie donde la condición es True y NaN donde es False. Por lo tanto, minutos_txt.where(~es_mmss) devuelve los valores de minutos_txt donde no es mm:ss y NaN donde sí lo es. Luego, pd.to_numeric() convierte esos valores a números, y los NaN se mantienen como NaN.
#luego, pd.to_numeric() convierte esos valores a números, y los NaN se mantienen como NaN.

#TERCERO: convierto las filas mm:ss a decimal
# 3) Las filas mm:ss se parten por los dos puntos: minutos + segundos/60
#--------------------------------------------------------------------------------
#HA DADO ERROR: EXPLICACIÓN ABAJO
######partes = minutos_txt[es_mmss].str.split(":", expand=True).astype(float)
######minutos[es_mmss] = partes[0] + partes[1] / 60
#--------------------------------------------------------------------------------
if es_mmss.any():
    partes = minutos_txt[es_mmss].str.split(":", expand=True).astype(float)
    minutos[es_mmss] = partes[0] + partes[1] / 60

player_stats["numMinutes"] = minutos.astype(float)

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Minutos en formato mm:ss convertidos a decimal", int(es_mmss.sum()), "Se convierten, no se eliminan"))

print("Filas convertidas de mm:ss:", es_mmss.sum())
player_stats["numMinutes"].describe()

Filas convertidas de mm:ss: 0


count    301653.000000
mean         22.240825
std          10.741454
min           0.000000
25%          14.530000
50%          23.000000
75%          31.000000
max          64.000000
Name: numMinutes, dtype: float64

**Lectura del `describe()` de los minutos:**

- **`count` 301.653 de 366.218 filas.** Las 64.565 filas restantes tienen los minutos vacíos: son jugadores que estaban convocados pero **no jugaron** (lesión, decisión del entrenador, otros motivos). No es un error de los datos. Los rellenaré con 0 más adelante, porque ese día su carga fue cero, y el motivo lo sacaré de la columna `comment`.
- **Media 22,2 y mediana 23 minutos** entre quienes jugaron. Es un valor razonable para la NBA, donde los titulares juegan unos 30-36 minutos y los suplentes bastante menos.
- **Máximo 64 minutos.** Es plausible: un partido dura 48 minutos y cada prórroga añade 5, así que 64 corresponde a un partido con tres prórrogas. No lo trato como valor atípico.
- **Mínimo 0.** Hay jugadores que constan como jugados con 0 minutos (por ejemplo, salieron solo unos segundos). Los mantengo.
- **Conclusión:** tras la conversión, `numMinutes` es numérica y sus valores están dentro de lo posible en un partido, así que la puedo usar para calcular la carga de 14 días.

ERROR ENCOTRADO Y SOLICIONADO:

**Minutos en formato mm:ss:** en la exploración vi que algunas filas guardaban los minutos como texto `mm:ss` (por ejemplo `34:12`) en lugar de como número decimal. Al aplicar la conversión, la celda dio un error (`KeyError: 0`). El motivo es que **en mi ventana (2015-16 → 2025-26) no hay ninguna fila en formato mm:ss**: todas eran de temporadas antiguas y ya las había quitado el filtro de fechas. Al no haber filas que convertir, `split(":")` devolvía una tabla sin columnas, y el código no encontraba la columna de los minutos (`partes[0]`).

**Corrección:** la conversión solo se hace si existe al menos una fila con ":" (`if es_mmss.any():`). Mantengo el paso en lugar de borrarlo, como con `personId`: si una versión futura del dataset trae minutos en mm:ss, se convertirán a decimal

### 3. CLASIFICAR COMMENT

Mi target lesion_nueva depende de saber por qué no jugó alguien. Esa información solo está en `comment`, que es de texto libre. Lo voy a convertir en categorías para poder trabajar con ello. 

In [12]:
#PRIMERO: nomralizo los comentarios de los jugadores para poder clasificar el motivo
#Paso el comentario a minúsculas y quito las comillas para buscar palabras sin problemas
texto = player_stats["comment"].astype("string").str.lower().str.replace('"', "", regex=False)

#SEGUNDO: creo una lista de palabras que indican lesión o enfermedad para poder clasificar el motivo de la ausencia
#Palabras que indican lesión o enfermedad (el | significa "o")
palabras_lesion = (
    "injur|illness|sprain|strain|sore|contusion|fractur|surgery|concussion|tendin|"
    "torn|tear|bruise|knee|ankle|back|hamstring|foot|hip|wrist|calf|groin|shoulder|"
    "achilles|thumb|finger|toe|hand|elbow|quad|neck|protocol|sick"
)

#TERCERO: clasificar el motivo de la ausencia según el comentario del jugador
#Clasifico cada fila; np.select comprueba las condiciones EN ORDEN y se queda con la primera que se cumple
player_stats["motivo_ausencia"] = np.select(
    [
        texto.isna(),                                              #comentario vacío, no se sabe el motivo
        texto.str.contains("health and safety|covid", na=False),   # NUEVO: protocolo COVID, no es lesión por carga
        texto.str.contains(palabras_lesion, na=False),             #lesión o enfermedad
        texto.str.contains("coach|rest", na=False),                #decisión técnica del entrenador
    ],
    ["jugó", "otros", "lesion", "decision_tecnica"],
    default="otros",
)
print(player_stats["motivo_ausencia"].value_counts())

motivo_ausencia
jugó                307623
decision_tecnica     49550
lesion                8142
otros                  903
Name: count, dtype: int64


In [13]:
#Comprobación de que la clasificación es correcta.
for categoria in ["lesion", "decision_tecnica", "otros"]:
    print(f"\n=== {categoria} ===")
    print(player_stats.loc[player_stats["motivo_ausencia"] == categoria, "comment"].value_counts().head(8))


=== lesion ===
comment
DND - Injury/Illness         4396
DNP - Injury/Illness         1739
NWT - Injury/Illness          300
NWT - Illness                  34
DND - Sprained Left Ankle      30
DND - Right Knee Soreness      29
DND - Left Ankle Sprain        29
DND - Illness                  24
Name: count, dtype: int64

=== decision_tecnica ===
comment
DNP - Coach's Decision    49200
DND - Rest                  201
NWT - Rest                   60
DNP - Rest                   33
DND - Coach's Decision       27
DND - rest                   10
NWT - Coach's Decision        6
DND_COACH                     5
Name: count, dtype: int64

=== otros ===
comment
NWT - Not With Team                  177
NWT - Personal                       117
NWT - League Suspension               68
DNP - League Suspension               55
NWT - Suspended                       53
NWT_LEAGUE_SUSPENSION                 44
NWT - Health and Safety Protocols     38
NWT - Personal Reasons                34
Name: count

**Cómo he clasificado los motivos de ausencia**

La columna `comment` solo tiene texto cuando un jugador no juega, y explica por qué. Como es texto libre, lo he agrupado en cuatro categorías buscando palabras clave:

- **jugó:** no hay comentario, así que el jugador disputó el partido.
- **lesion:** el comentario habla de una lesión o enfermedad (por ejemplo "Injury/Illness", "Ankle Sprain" o "Knee Soreness").
- **decision_tecnica:** el entrenador decidió no sacarlo ("Coach's Decision") o le dio descanso ("Rest").
- **otros:** cualquier otro motivo, como sanciones, motivos personales o el protocolo COVID.

La categoría que más me importa es **lesion**, porque de ahí saco el target del modelo.

**Por qué he sacado el COVID de las lesiones**

Al revisar los comentarios vi que "Health and Safety Protocols" (el protocolo COVID de 2020-2022) se estaba contando como lesión. Lo he movido a "otros" porque mi proyecto intenta comprobar si **jugar muchos minutos aumenta el riesgo de lesión**, y un jugador no se contagia de COVID por jugar más. Si lo dejaba como lesión, el modelo aprendería de casos que no tienen nada que ver con la carga y los resultados serían menos fiables.

**Lo que he comprobado**

He mirado los comentarios más repetidos de cada categoría para asegurarme de que cada uno está donde toca. En "lesion" solo hay lesiones y enfermedades, y en "otros" no se ha colado ninguna lesión.

**Limitaciones que tengo en cuenta**

- La mayoría de lesiones solo pone "Injury/Illness", sin decir qué lesión es. Por eso no analizo los tipos de lesión.
- Cuando un equipo da descanso a un jugador para cuidarlo, casi siempre lo apunta como "Coach's Decision". No puedo distinguir ese descanso de una decisión técnica normal, pero no afecta a mi target, porque solo uso las lesiones.
- Hay unas 6.000 filas sin minutos y sin comentario: el jugador no jugó y no se sabe por qué. Si alguna fue una lesión, no la detecto, así que es posible que haya algunas lesiones más de las que cuento.

### 4. MINUTOS DE QUIEN *NO* JUGÓ

In [14]:
# Filas sin minutos: el jugador estaba convocado pero no jugó (con o sin comentario)
sin_minutos = player_stats["numMinutes"].isna()

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Sin minutos (no jugó): se rellenan con 0", int(sin_minutos.sum()),
                "El jugador no disputó el partido, su carga ese día fue 0"))

player_stats["numMinutes"] = player_stats["numMinutes"].fillna(0)
print("Filas rellenadas con 0:", sin_minutos.sum())
print("Nulos que quedan en numMinutes:", player_stats["numMinutes"].isna().sum())

Filas rellenadas con 0: 64565
Nulos que quedan en numMinutes: 0


Se rellena con 0 por qué si se rellenara con la media (22 min), estaría inventando que ese jugador jugó y su carga de 14 días saldría más alta de lo real. 

Si borrara las filas, perdería justo los partidos de ausencia, que son de donde sale el target -> `lesion_nueva`.

0 es la verdad, porque ese día no sumó minutos. El **motivo** no se pierde, porque ya está en `motivo_ausencia`. 

### 5. EDAD DEL JUGADOR

La edad no está en PlayerStatistics.csvm pero sí la fecha de nacimiento en Players.csv. Hay que unir las dos tablas por `personId` y calcular la edad en la fecha de cada partido. 

In [19]:
#PRIMERO:Convierto la fecha de nacimiento a fecha
players["birthDate"] = pd.to_datetime(players["birthDate"], errors="coerce")

#SEGUNDO
#Uno las tablas: a cada fila de partido le añado la fecha de nacimiento del jugador
n = len(player_stats)
# Si ya ejecuté esta celda antes, borro las columnas que creó, para no duplicarlas
player_stats = player_stats.drop(columns=["birthDate", "birthDate_x", "birthDate_y", "edad"], errors="ignore")
player_stats = player_stats.merge(players[["personId", "birthDate"]], on="personId", how="left")
print("Filas antes:", n, "| después:", len(player_stats))   # deben ser IGUALES
#Con el how="left" me aseguro de que se mantengan todas las filas de player_stats, aunque no haya coincidencia en players.
#Si no hay coincidencia, la columna birthDate se rellenará con NaT (Not a Time).

#TERCERO
#Edad el día del partido, en años con decimales
player_stats["edad"] = (player_stats["gameDate"] - player_stats["birthDate"]).dt.days / 365.25


#No se borran las filas sin edad. La edad es solo una variable más y el modelo puede rellenar las que falten con la mediana.
#Si se borraran las filas sin edad, se perderían datos de partidos y jugadores que sí tienen edad, lo que podría afectar negativamente al análisis.
sin_edad = player_stats["edad"].isna().sum()

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Sin fecha de nacimiento (edad vacía)", int(sin_edad),
                "Se mantienen; el modelo rellenará la edad que falte"))


print("Filas sin edad:", sin_edad)
player_stats["edad"].describe()

Filas antes: 366218 | después: 366218
Filas sin edad: 11


count    366207.000000
mean         26.707031
std           4.346688
min          18.814511
25%          23.389459
50%          25.995893
75%          29.541410
max          43.121150
Name: edad, dtype: float64

**Minutos de quien no jugó:** relleno con 0 los minutos vacíos, porque ese día el jugador no jugó y su carga fue 0. No uso la media, porque inventaría minutos que no jugó, y no borro las filas, porque son justo las ausencias de donde saco el target. El motivo de la ausencia no se pierde: está en `motivo_ausencia`.

**Edad:** la fecha de nacimiento está en `Players.csv`, así que uno las dos tablas por `personId` (con `how="left"`, para no perder partidos) y calculo la edad el día de cada partido. Uso la edad de cada partido y no la de la temporada porque mi predicción es partido a partido, y así es más exacta. La redondeo a un decimal para que se lea mejor. Las filas sin fecha de nacimiento se mantienen y el modelo rellenará esa edad.

**Celda repetible:** al ejecutar la celda varias veces, el merge duplicaba la columna `birthDate` y daba error. Lo he solucionado borrando antes las columnas que crea la celda, para que se pueda ejecutar las veces que haga falta.